# DuckDB: In-Process Analytical SQL

DuckDB is an **in-process** analytical database. It runs inside your Python process: no server to install, no connection string to configure, no daemon to start. You import it, and it is ready to run SQL.

**Why DuckDB matters in 2026:**
- Query CSV, Parquet, pandas DataFrames, and Polars DataFrames directly with standard SQL
- Vectorized, columnar execution designed for analytics (OLAP), not row-by-row transactions
- Replaces Spark for local analytical workloads that previously required a cluster
- Seamlessly interoperates with the rest of the Python data stack via Apache Arrow

**Core idea:** `duckdb.sql("SELECT * FROM 'data.parquet' WHERE year > 2020").df()` -- one line reads a Parquet file and returns a pandas DataFrame, with no setup.

In [1]:
import duckdb
import pandas as pd
import polars as pl
import numpy as np

# In-memory connection (default: no persistence)
con = duckdb.connect()

print(f"DuckDB version: {duckdb.__version__}")
print(f"Connection type: {type(con)}")

# Quick sanity check
result = con.execute("SELECT 42 AS answer, 'DuckDB is ready' AS status").fetchdf()
print(result)

DuckDB version: 1.5.4
Connection type: <class '_duckdb.DuckDBPyConnection'>
   answer           status
0      42  DuckDB is ready


## 1. Basic SQL on In-Memory Data

DuckDB supports the full SQL standard including CTEs, window functions, UNNEST, and many analytical extensions. You can run SQL directly on any registered table or on Python variables using the `FROM python_var` syntax.

Key methods:
- `con.execute("SQL").fetchdf()` returns a pandas DataFrame
- `con.sql("SQL").df()` is shorthand for the same thing
- `con.execute("SQL").fetchall()` returns a list of tuples

In [2]:
# Create a table in DuckDB and run SQL
con.execute("""
    CREATE OR REPLACE TABLE employees (
        emp_id     INTEGER,
        name       VARCHAR,
        department VARCHAR,
        salary     DOUBLE,
        hire_year  INTEGER,
        score      DOUBLE
    )
""")

con.execute("""
    INSERT INTO employees VALUES
        (1, 'Alice',  'Engineering', 72000, 2019, 88.5),
        (2, 'Bob',    'Sales',       95000, 2017, 76.2),
        (3, 'Carol',  'Engineering', 61000, 2021, 91.0),
        (4, 'David',  'Management', 110000, 2015, 84.3),
        (5, 'Eve',    'Sales',       83000, 2020, 79.8),
        (6, 'Frank',  'Engineering', 98000, 2018, 93.1),
        (7, 'Grace',  'HR',          57000, 2022, 70.5),
        (8, 'Heidi',  'HR',          62000, 2021, 75.0),
        (9, 'Ivan',   'Sales',       89000, 2019, 82.4),
        (10,'Judy',   'Engineering', 105000, 2016, 95.0)
""")

print("All employees:")
print(con.execute("SELECT * FROM employees ORDER BY emp_id").fetchdf())

print("\nFiltered and sorted:")
print(con.execute("""
    SELECT name, department, salary, score
    FROM employees
    WHERE salary > 80000
    ORDER BY salary DESC
""").fetchdf())

All employees:
   emp_id   name   department    salary  hire_year  score
0       1  Alice  Engineering   72000.0       2019   88.5
1       2    Bob        Sales   95000.0       2017   76.2
2       3  Carol  Engineering   61000.0       2021   91.0
3       4  David   Management  110000.0       2015   84.3
4       5    Eve        Sales   83000.0       2020   79.8
5       6  Frank  Engineering   98000.0       2018   93.1
6       7  Grace           HR   57000.0       2022   70.5
7       8  Heidi           HR   62000.0       2021   75.0
8       9   Ivan        Sales   89000.0       2019   82.4
9      10   Judy  Engineering  105000.0       2016   95.0

Filtered and sorted:
    name   department    salary  score
0  David   Management  110000.0   84.3
1   Judy  Engineering  105000.0   95.0
2  Frank  Engineering   98000.0   93.1
3    Bob        Sales   95000.0   76.2
4   Ivan        Sales   89000.0   82.4
5    Eve        Sales   83000.0   79.8


## 2. Query pandas and Polars DataFrames Directly with SQL

DuckDB's killer feature for Python users: you can reference **Python variables** directly in SQL queries using the variable name. DuckDB reads the Arrow memory of the DataFrame without copying data.

In [3]:
# Create a pandas DataFrame
pandas_df = pd.DataFrame({
    "product":  ["Widget A", "Widget B", "Gadget X", "Gadget Y", "Widget A", "Gadget X"],
    "region":   ["North", "South", "North", "East", "East", "South"],
    "quantity": [120, 85, 200, 150, 95, 175],
    "price":    [29.99, 29.99, 49.99, 49.99, 29.99, 49.99],
})

# Query the pandas DataFrame directly - reference it by variable name
print("Query pandas DataFrame with SQL:")
result = con.execute("""
    SELECT
        product,
        SUM(quantity) AS total_qty,
        ROUND(SUM(quantity * price), 2) AS total_revenue
    FROM pandas_df
    GROUP BY product
    ORDER BY total_revenue DESC
""").fetchdf()
print(result)

# Create a Polars DataFrame
polars_df = pl.DataFrame({
    "category": ["A", "A", "B", "B", "C"],
    "value":    [10.0, 20.0, 30.0, 40.0, 50.0],
    "weight":   [1, 2, 3, 4, 5],
})

# Query the Polars DataFrame directly
print("\nQuery Polars DataFrame with SQL:")
result2 = con.execute("""
    SELECT
        category,
        SUM(value) AS total_value,
        AVG(weight) AS avg_weight,
        COUNT(*) AS row_count
    FROM polars_df
    GROUP BY category
    ORDER BY category
""").fetchdf()
print(result2)

Query pandas DataFrame with SQL:
    product  total_qty  total_revenue
0  Gadget X      375.0       18746.25
1  Gadget Y      150.0        7498.50
2  Widget A      215.0        6447.85
3  Widget B       85.0        2549.15

Query Polars DataFrame with SQL:
  category  total_value  avg_weight  row_count
0        A         30.0         1.5          2
1        B         70.0         3.5          2
2        C         50.0         5.0          1


## 3. Read CSV and Parquet Files Directly with SQL

DuckDB can read files directly in SQL using `FROM 'path/to/file.csv'` or `FROM 'path/to/file.parquet'`. No loading step required. DuckDB reads only the columns and rows needed to satisfy the query.

In [4]:
import tempfile, os

# Write sample files
csv_path     = "/tmp/duckdb_demo.csv"
parquet_path = "/tmp/duckdb_demo.parquet"

sample_df = pd.DataFrame({
    "year":       [2021, 2021, 2022, 2022, 2023, 2023, 2023],
    "quarter":    ["Q1", "Q2", "Q1", "Q3", "Q1", "Q2", "Q4"],
    "department": ["Sales", "Sales", "Eng", "Eng", "Sales", "Eng", "HR"],
    "revenue":    [450000, 520000, 380000, 410000, 610000, 490000, 220000],
    "costs":      [300000, 350000, 200000, 210000, 400000, 260000, 150000],
})
sample_df.to_csv(csv_path, index=False)
sample_df.to_parquet(parquet_path, index=False)

# Query CSV directly
print("Query CSV file directly:")
print(con.execute(f"""
    SELECT year, department, SUM(revenue) AS total_revenue
    FROM '{csv_path}'
    WHERE year >= 2022
    GROUP BY year, department
    ORDER BY year, total_revenue DESC
""").fetchdf())

# Query Parquet directly
print("\nQuery Parquet file directly:")
print(con.execute(f"""
    SELECT
        year,
        SUM(revenue - costs) AS total_profit,
        ROUND(SUM(revenue - costs) * 100.0 / SUM(revenue), 1) AS profit_margin_pct
    FROM '{parquet_path}'
    GROUP BY year
    ORDER BY year
""").fetchdf())

Query CSV file directly:
   year department  total_revenue
0  2022        Eng       790000.0
1  2023      Sales       610000.0
2  2023        Eng       490000.0
3  2023         HR       220000.0

Query Parquet file directly:
   year  total_profit  profit_margin_pct
0  2021      320000.0               33.0
1  2022      380000.0               48.1
2  2023      510000.0               38.6


## 4. Aggregate Queries: GROUP BY, HAVING, and Window Functions

DuckDB supports the full SQL analytics toolkit including:
- `GROUP BY` with `HAVING` for post-aggregation filtering
- All standard window functions: `ROW_NUMBER()`, `RANK()`, `DENSE_RANK()`, `LAG()`, `LEAD()`, `SUM() OVER (...)`, `AVG() OVER (...)`
- `QUALIFY` clause (DuckDB extension) for filtering on window function results without a subquery

In [5]:
# GROUP BY with HAVING
print("Departments with average salary above 80000:")
print(con.execute("""
    SELECT
        department,
        COUNT(*)          AS headcount,
        ROUND(AVG(salary), 0) AS avg_salary,
        MIN(salary)       AS min_salary,
        MAX(salary)       AS max_salary
    FROM employees
    GROUP BY department
    HAVING AVG(salary) > 80000
    ORDER BY avg_salary DESC
""").fetchdf())

# Window functions in SQL
print("\nEmployee rank within department by salary:")
print(con.execute("""
    SELECT
        name,
        department,
        salary,
        RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS dept_rank,
        ROUND(AVG(salary) OVER (PARTITION BY department), 0)       AS dept_avg_salary,
        SUM(salary)  OVER (PARTITION BY department)                AS dept_total_salary
    FROM employees
    ORDER BY department, dept_rank
""").fetchdf())

# QUALIFY to filter on window results
print("\nTop earner per department (QUALIFY):")
print(con.execute("""
    SELECT name, department, salary
    FROM employees
    QUALIFY RANK() OVER (PARTITION BY department ORDER BY salary DESC) = 1
    ORDER BY department
""").fetchdf())

Departments with average salary above 80000:
    department  headcount  avg_salary  min_salary  max_salary
0   Management          1    110000.0    110000.0    110000.0
1        Sales          3     89000.0     83000.0     95000.0
2  Engineering          4     84000.0     61000.0    105000.0

Employee rank within department by salary:
    name   department    salary  dept_rank  dept_avg_salary  dept_total_salary
0   Judy  Engineering  105000.0          1          84000.0           336000.0
1  Frank  Engineering   98000.0          2          84000.0           336000.0
2  Alice  Engineering   72000.0          3          84000.0           336000.0
3  Carol  Engineering   61000.0          4          84000.0           336000.0
4  Heidi           HR   62000.0          1          59500.0           119000.0
5  Grace           HR   57000.0          2          59500.0           119000.0
6  David   Management  110000.0          1         110000.0           110000.0
7    Bob        Sales   95000.0

## 5. DuckDB vs SQLite: OLAP vs OLTP

Both SQLite and DuckDB are embedded databases that require no server. But they are designed for completely different workloads:

| Dimension | SQLite | DuckDB |
|-----------|--------|--------|
| Workload type | OLTP (transactional) | OLAP (analytical) |
| Execution model | Row-by-row | Columnar, vectorized |
| Best for | Many small reads and writes | Large aggregations on many rows |
| Concurrent writes | Supports concurrent readers, one writer | Append-friendly, optimized for reads |
| Analytics performance | Slow on aggregating millions of rows | Fast (10 to 100x vs SQLite for aggregations) |
| Python file I/O | Read `sqlite3` files | Read CSV, Parquet, Arrow, JSON |

**Rule of thumb:** Use SQLite for apps that store and retrieve records. Use DuckDB for data analysis.

In [6]:
import sqlite3, time

# Create a large dataset
N = 500_000
rng = np.random.default_rng(42)
depts = ["Sales", "Eng", "HR", "Finance", "Marketing"]

large_data = pd.DataFrame({
    "department": rng.choice(depts, N),
    "salary":     rng.integers(40000, 150000, N).astype(float),
    "score":      rng.uniform(50, 100, N),
})

# SQLite benchmark
sqlite_path = "/tmp/bench.sqlite"
if os.path.exists(sqlite_path):
    os.remove(sqlite_path)
conn_sqlite = sqlite3.connect(sqlite_path)
large_data.to_sql("data", conn_sqlite, index=False, if_exists="replace")

t0 = time.perf_counter()
pd.read_sql("""
    SELECT department, AVG(salary), AVG(score), COUNT(*)
    FROM data
    GROUP BY department
""", conn_sqlite)
sqlite_time = time.perf_counter() - t0
conn_sqlite.close()

# DuckDB benchmark (on the same pandas DataFrame)
t0 = time.perf_counter()
con.execute("""
    SELECT department, AVG(salary), AVG(score), COUNT(*)
    FROM large_data
    GROUP BY department
""").fetchdf()
duckdb_time = time.perf_counter() - t0

speedup = sqlite_time / duckdb_time
print(f"Dataset: {N:,} rows")
print(f"SQLite:  {sqlite_time:.4f}s")
print(f"DuckDB:  {duckdb_time:.4f}s")
print(f"DuckDB is {speedup:.1f}x faster for this aggregation")

Dataset: 500,000 rows
SQLite:  0.3303s
DuckDB:  0.0113s
DuckDB is 29.2x faster for this aggregation


## 6. Export to Parquet with SQL COPY

DuckDB can write query results directly to Parquet files using the `COPY ... TO` SQL statement. This is useful for building ETL pipelines that read raw data, transform it with SQL, and write the results as Parquet for downstream consumers.

In [7]:
output_parquet = "/tmp/duckdb_output.parquet"
output_csv     = "/tmp/duckdb_output.csv"

# Export a SQL query result to Parquet
con.execute(f"""
    COPY (
        SELECT
            department,
            COUNT(*)              AS headcount,
            ROUND(AVG(salary), 2) AS avg_salary,
            ROUND(AVG(score), 2)  AS avg_score,
            MIN(hire_year)        AS earliest_hire
        FROM employees
        GROUP BY department
        ORDER BY avg_salary DESC
    ) TO '{output_parquet}' (FORMAT PARQUET)
""")
print(f"Exported to Parquet: {output_parquet}")
print(f"File size: {os.path.getsize(output_parquet):,} bytes")

# Export to CSV
con.execute(f"""
    COPY (
        SELECT * FROM employees WHERE salary > 80000
    ) TO '{output_csv}' (HEADER, DELIMITER ',')
""")
print(f"Exported to CSV: {output_csv}")

# Read the exported Parquet back as a pandas DataFrame
print("\nRead exported Parquet:")
print(con.execute(f"SELECT * FROM '{output_parquet}'").fetchdf())

Exported to Parquet: /tmp/duckdb_output.parquet
File size: 861 bytes
Exported to CSV: /tmp/duckdb_output.csv

Read exported Parquet:
    department  headcount  avg_salary  avg_score  earliest_hire
0   Management          1    110000.0      84.30           2015
1        Sales          3     89000.0      79.47           2017
2  Engineering          4     84000.0      91.90           2016
3           HR          2     59500.0      72.75           2021


## 7. DuckDB with Pandas: The .df() Pattern

The most common DuckDB pattern in Python notebooks is the `.df()` chain: run SQL, get a pandas DataFrame back. This lets you use DuckDB for heavy aggregation and pandas for visualization or ML integration.

Result format options:
- `.fetchdf()` or `.df()` -- pandas DataFrame
- `.pl()` -- Polars DataFrame (zero-copy via Arrow)
- `.fetchall()` -- list of tuples
- `.fetchone()` -- single row tuple
- `.arrow()` -- PyArrow Table

In [8]:
# The .df() shorthand
summary = con.sql("""
    SELECT
        department,
        COUNT(*)              AS headcount,
        ROUND(AVG(salary), 0) AS avg_salary,
        MAX(salary)           AS top_salary
    FROM employees
    GROUP BY department
    ORDER BY avg_salary DESC
""").df()   # returns a pandas DataFrame

print("Result as pandas DataFrame (.df()):")
print(summary)
print(f"Type: {type(summary)}")

# .pl() returns a Polars DataFrame directly
polars_result = con.sql("""
    SELECT name, salary, score
    FROM employees
    ORDER BY score DESC
    LIMIT 5
""").pl()

print("\nResult as Polars DataFrame (.pl()):")
print(polars_result)
print(f"Type: {type(polars_result)}")

# .arrow() returns a PyArrow Table
arrow_result = con.sql("SELECT * FROM employees LIMIT 3").arrow()
print(f"\nArrow Table schema: {arrow_result.schema}")

Result as pandas DataFrame (.df()):
    department  headcount  avg_salary  top_salary
0   Management          1    110000.0    110000.0
1        Sales          3     89000.0     95000.0
2  Engineering          4     84000.0    105000.0
3           HR          2     59500.0     62000.0
Type: <class 'pandas.core.frame.DataFrame'>

Result as Polars DataFrame (.pl()):
shape: (5, 3)
┌───────┬──────────┬───────┐
│ name  ┆ salary   ┆ score │
│ ---   ┆ ---      ┆ ---   │
│ str   ┆ f64      ┆ f64   │
╞═══════╪══════════╪═══════╡
│ Judy  ┆ 105000.0 ┆ 95.0  │
│ Frank ┆ 98000.0  ┆ 93.1  │
│ Carol ┆ 61000.0  ┆ 91.0  │
│ Alice ┆ 72000.0  ┆ 88.5  │
│ David ┆ 110000.0 ┆ 84.3  │
└───────┴──────────┴───────┘
Type: <class 'polars.dataframe.frame.DataFrame'>

Arrow Table schema: emp_id: int32
name: string
department: string
salary: double
hire_year: int32
score: double


## 8. Performance Demo: 1M Row Aggregation vs Pandas GroupBy

DuckDB's vectorized columnar execution makes it significantly faster than pandas for aggregation on large datasets. This benchmark runs a multi-column groupby on 1 million rows and compares wall clock time.

In [9]:
import time

N = 1_000_000
rng = np.random.default_rng(42)
departments = ["Engineering", "Sales", "Marketing", "HR", "Finance"]
regions     = ["North", "South", "East", "West"]

big_data = pd.DataFrame({
    "department": rng.choice(departments, N),
    "region":     rng.choice(regions, N),
    "salary":     rng.integers(40000, 150000, N).astype(float),
    "score":      rng.uniform(50, 100, N),
    "tenure":     rng.integers(1, 20, N).astype(float),
})

# Pandas benchmark
t0 = time.perf_counter()
_ = big_data.groupby(["department", "region"]).agg(
    avg_salary=("salary", "mean"),
    total_salary=("salary", "sum"),
    avg_score=("score", "mean"),
    count=("salary", "count")
).reset_index()
pandas_time = time.perf_counter() - t0

# DuckDB benchmark (references big_data in-scope)
t0 = time.perf_counter()
_ = con.execute("""
    SELECT
        department,
        region,
        AVG(salary)   AS avg_salary,
        SUM(salary)   AS total_salary,
        AVG(score)    AS avg_score,
        COUNT(*)      AS cnt
    FROM big_data
    GROUP BY department, region
    ORDER BY department, region
""").fetchdf()
duckdb_time = time.perf_counter() - t0

speedup = pandas_time / duckdb_time
print(f"Dataset: {N:,} rows, 5 departments x 4 regions = 20 groups")
print(f"Pandas:  {pandas_time:.4f}s")
print(f"DuckDB:  {duckdb_time:.4f}s")
print(f"Speedup: {speedup:.1f}x faster")

Dataset: 1,000,000 rows, 5 departments x 4 regions = 20 groups
Pandas:  0.1482s
DuckDB:  0.0232s
Speedup: 6.4x faster


## 9. Persistent Database: Saving DuckDB to Disk

By default, `duckdb.connect()` creates an in-memory database. To persist data across sessions, pass a file path. The `.duckdb` file stores the database, tables, and indexes. You can reconnect to it later and all tables will be there.

In [10]:
db_path = "/tmp/mydb.duckdb"

# Create a persistent connection
persistent_con = duckdb.connect(db_path)

# Create and populate a table
persistent_con.execute("""
    CREATE OR REPLACE TABLE projects (
        project_id   INTEGER,
        project_name VARCHAR,
        start_date   DATE,
        budget       DOUBLE,
        status       VARCHAR
    )
""")

persistent_con.execute("""
    INSERT INTO projects VALUES
        (1, 'Alpha',   '2024-01-15', 500000, 'Active'),
        (2, 'Beta',    '2024-03-01', 250000, 'Complete'),
        (3, 'Gamma',   '2025-06-10', 750000, 'Active'),
        (4, 'Delta',   '2025-09-01', 180000, 'Planning'),
        (5, 'Epsilon', '2026-01-01', 920000, 'Planning')
""")

print("Tables in the persistent database:")
print(persistent_con.execute("SHOW TABLES").fetchdf())

print("\nProject data:")
print(persistent_con.execute("SELECT * FROM projects ORDER BY start_date").fetchdf())

persistent_con.close()

# Reconnect and verify data persists
persistent_con2 = duckdb.connect(db_path)
print("\nAfter reconnect, data still available:")
print(persistent_con2.execute("""
    SELECT status, COUNT(*) AS n, SUM(budget) AS total_budget
    FROM projects
    GROUP BY status
""").fetchdf())
persistent_con2.close()
print(f"\nDatabase file: {db_path} ({os.path.getsize(db_path):,} bytes)")

Tables in the persistent database:
       name
0  projects

Project data:
   project_id project_name start_date    budget    status
0           1        Alpha 2024-01-15  500000.0    Active
1           2         Beta 2024-03-01  250000.0  Complete
2           3        Gamma 2025-06-10  750000.0    Active
3           4        Delta 2025-09-01  180000.0  Planning
4           5      Epsilon 2026-01-01  920000.0  Planning

After reconnect, data still available:
     status  n  total_budget
0    Active  2     1250000.0
1  Complete  1      250000.0
2  Planning  2     1100000.0

Database file: /tmp/mydb.duckdb (536,576 bytes)


## 10. CTEs and Complex Analytical SQL

DuckDB supports Common Table Expressions (CTEs), recursive CTEs, and complex multi-step SQL that would require multiple pandas operations. CTEs make analytical SQL readable and maintainable.

In [11]:
# Multi-step CTE: department stats then percentile classification
result = con.execute("""
    WITH dept_stats AS (
        SELECT
            department,
            AVG(salary) AS dept_avg_salary,
            STDDEV(salary) AS dept_std_salary,
            COUNT(*) AS headcount
        FROM employees
        GROUP BY department
    ),
    employee_z_scores AS (
        SELECT
            e.name,
            e.department,
            e.salary,
            d.dept_avg_salary,
            ROUND((e.salary - d.dept_avg_salary) / NULLIF(d.dept_std_salary, 0), 2) AS z_score
        FROM employees e
        JOIN dept_stats d USING (department)
    )
    SELECT
        name,
        department,
        salary,
        z_score,
        CASE
            WHEN z_score > 0.5  THEN 'Above Average'
            WHEN z_score < -0.5 THEN 'Below Average'
            ELSE 'Average'
        END AS salary_band
    FROM employee_z_scores
    ORDER BY department, z_score DESC
""").fetchdf()

print("Employee salary Z-scores within department:")
print(result)

Employee salary Z-scores within department:
    name   department    salary  z_score    salary_band
0   Judy  Engineering  105000.0     1.00  Above Average
1  Frank  Engineering   98000.0     0.67  Above Average
2  Alice  Engineering   72000.0    -0.57  Below Average
3  Carol  Engineering   61000.0    -1.10  Below Average
4  Heidi           HR   62000.0     0.71  Above Average
5  Grace           HR   57000.0    -0.71  Below Average
6  David   Management  110000.0      NaN        Average
7    Bob        Sales   95000.0     1.00  Above Average
8   Ivan        Sales   89000.0     0.00        Average
9    Eve        Sales   83000.0    -1.00  Below Average


## Key Takeaways

1. **In-process, no server**: `import duckdb; con = duckdb.connect()` is all the setup required. DuckDB runs inside your Python process.

2. **SQL on everything**: Query pandas DataFrames, Polars DataFrames, CSV files, and Parquet files with the same SQL syntax. Reference Python variables by name directly in SQL.

3. **OLAP, not OLTP**: DuckDB is designed for analytical queries (aggregations, scans, joins on many rows). Use SQLite for transactional workloads (inserts, updates, concurrent writers).

4. **Return format is flexible**: `.df()` returns pandas, `.pl()` returns Polars, `.arrow()` returns PyArrow. Zero-copy when possible via Apache Arrow.

5. **Parquet is the native format**: DuckDB reads and writes Parquet natively. The combination of DuckDB + Parquet files is the modern alternative to heavy warehouse setups for local analytics.

6. **When to use DuckDB**: ad-hoc exploration of CSV or Parquet files, replacing pandas groupby on large datasets, building SQL-based ETL pipelines, and any time you want the expressiveness of SQL without a server.

**The modern local data stack: Polars for DataFrame transformations, DuckDB for SQL analytics, Parquet for storage. All three are interoperable via Apache Arrow.**